# 04 — Advanced Analysis: Clustering & PCA

Tạo digital lifestyle archetypes, PCA user map, density contour và parallel coordinates.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

# Paths
PROJECT_ROOT = Path("..")
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "mobile_usage_behavioral_analysis.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

# Create necessary directories
for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
def title(t):
    print("\n" + "="*90); print(t); print("="*90)
def clean_cols(df):
    df = df.copy(); df.columns = [c.strip() for c in df.columns]; return df

# Load dataset and basic audit
title("Load processed dataset")
df = pd.read_csv(PROCESSED_DIR / "smartphone_features.csv")
print(df.shape)
display(df.head())


In [ ]:
# Prepare modeling matrix
title("Prepare modeling matrix")
features = ["Age", "Daily_Screen_Time_Hours", "Total_App_Usage_Hours", "Number_of_Apps_Used", "Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Entertainment_Hours", "App_Fragmentation_Score"]
X = df[features].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
display(X.describe().T.round(3))
print("X_scaled:", X_scaled.shape)


In [ ]:
# Evaluate cluster count
title("Evaluate cluster count")
rows = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    rows.append({"k": k, "Inertia": km.inertia_, "Silhouette": silhouette_score(X_scaled, labels)})
eval_df = pd.DataFrame(rows)
display(eval_df.round(4))

# Visualize evaluation metrics
px.line(eval_df, x="k", y="Inertia", markers=True, title="Elbow method").show()
px.line(eval_df, x="k", y="Silhouette", markers=True, title="Silhouette score").show()
print("For storytelling, k=4 is used to create interpretable digital lifestyle archetypes.")


In [ ]:
# Fit KMeans and name clusters
title("Fit KMeans and name clusters")
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
df["Cluster"] = kmeans.fit_predict(X_scaled)
profile = df.groupby("Cluster")[features].mean().round(2)
display(profile)

# Automatic cluster naming based on strongest behavioral trait.
cluster_names = {}
for cid, row in profile.iterrows():
    if row["Daily_Screen_Time_Hours"] < profile["Daily_Screen_Time_Hours"].median() and row["Number_of_Apps_Used"] < profile["Number_of_Apps_Used"].median():
        name = "Digital Minimalists"
    elif row["Social_Media_Usage_Hours"] >= max(row["Productivity_App_Usage_Hours"], row["Gaming_App_Usage_Hours"]):
        name = "Social Enthusiasts"
    elif row["Productivity_App_Usage_Hours"] >= max(row["Social_Media_Usage_Hours"], row["Gaming_App_Usage_Hours"]):
        name = "Productivity Focused"
    else:
        name = "Mobile Gamers"
    cluster_names[cid] = name
    
# avoid duplicate names
seen, final = {}, {}
for cid, name in cluster_names.items():
    seen[name] = seen.get(name, 0) + 1
    final[cid] = name if seen[name] == 1 else f"{name} {seen[name]}"
df["Cluster_Name"] = df["Cluster"].map(final)

display(df["Cluster_Name"].value_counts().to_frame("Users"))
display(df.groupby("Cluster_Name")[features].mean().round(2))


In [ ]:
# PCA user map
title("PCA user map")
pca = PCA(n_components=2, random_state=42)
pts = pca.fit_transform(X_scaled)
df["PCA_1"] = pts[:, 0]
df["PCA_2"] = pts[:, 1]
print(f"PCA 1 variance: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"PCA 2 variance: {pca.explained_variance_ratio_[1]*100:.2f}%")
fig = px.scatter(df, x="PCA_1", y="PCA_2", color="Cluster_Name", opacity=0.75, hover_data=["Age", "Gender", "Location", "Daily_Screen_Time_Hours"], title="PCA user map — behavioral space")
fig.update_layout(height=650)
fig.show()


In [ ]:
# Cluster profiles and advanced visual prototypes
title("Cluster profiles and advanced visual prototypes")
profile_cols = ["Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Daily_Screen_Time_Hours", "Total_App_Usage_Hours", "Number_of_Apps_Used"]
long_profile = df.groupby("Cluster_Name")[profile_cols].mean().reset_index().melt(id_vars="Cluster_Name", var_name="Metric", value_name="Average")
fig = px.bar(long_profile, x="Metric", y="Average", color="Cluster_Name", barmode="group", title="Cluster profile comparison")
fig.update_layout(xaxis_tickangle=-30, height=560)
fig.show()

# Explore interesting relationships within clusters
fig = px.density_contour(df, x="Daily_Screen_Time_Hours", y="Total_App_Usage_Hours", color="Cluster_Name", title="Density contours — screen time vs total app usage")
fig.update_traces(contours_coloring="fill", contours_showlabels=True)
fig.show()

# Parallel coordinates for high-dimensional behavior patterns
sample = df.sample(min(500, len(df)), random_state=42).copy()
codes = {n: i for i, n in enumerate(sorted(df["Cluster_Name"].unique()))}
sample["Cluster_Code"] = sample["Cluster_Name"].map(codes)
fig = px.parallel_coordinates(sample, dimensions=["Age", "Number_of_Apps_Used", "Daily_Screen_Time_Hours", "Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Total_App_Usage_Hours"], color="Cluster_Code", title="Parallel coordinates — high-dimensional user behavior")
fig.show()


In [ ]:
# Export advanced analytical dataset
title("Export advanced analytical dataset")
df.to_csv(PROCESSED_DIR / "smartphone_clustered.csv", index=False)
df.groupby("Cluster_Name")[profile_cols].mean().round(3).reset_index().to_csv(PROCESSED_DIR / "cluster_profiles.csv", index=False)
size = df["Cluster_Name"].value_counts().reset_index()
size.columns = ["Cluster_Name", "Users"]
size["Percentage"] = size["Users"] / len(df)
size.to_csv(PROCESSED_DIR / "cluster_sizes.csv", index=False)
print("Saved smartphone_clustered.csv, cluster_profiles.csv, cluster_sizes.csv")
